In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "enviroment_bj").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("project_root:", ROOT)

project_root: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl


In [5]:
from pathlib import Path
from inference.final_wrapper import comparison_models

CHECKPOINTS_DIR = Path(
    r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints"
)

COMMON_FF = {
    "architecture": "feedforward",
    "feedforward_hidden_dims": (256, 256, 128),
    "use_layer_norm": False,
    "use_phase_adapters": False,
    "use_module_gating": False,
}

checkpoint_specs = [
    {
        "name": "03C_playing_only_hard",
        "checkpoint_path": CHECKPOINTS_DIR / "stage_03c_unknown_progress_hard_from_stage03b_v1" / "best_eval.pt",
        "bet_multipliers": (1,),
        "axu_loss_bet": False,
        **COMMON_FF,
    },
    {
        "name": "04D_weighted_ce_1234",
        "checkpoint_path": CHECKPOINTS_DIR / "stage_04d_betting_aux_weighted_ce_from_stage03c_v1" / "best_eval.pt",
        "bet_multipliers": (1, 2, 3, 4),
        "axu_loss_bet": True,
        **COMMON_FF,
    },
    {
        "name": "04G_ev_calibrated_1234",
        "checkpoint_path": CHECKPOINTS_DIR / "stage_04g_betting_ev_calibrated_from_stage04d_v1" / "best_eval.pt",
        "bet_multipliers": (1, 2, 3, 4),
        "axu_loss_bet": True,
        **COMMON_FF,
    },
    {
        "name": "05A_count_aux_repr_1234",
        "checkpoint_path": CHECKPOINTS_DIR / "stage_05a_count_aux_representation_from_04g_v1" / "best_eval.pt",
        "bet_multipliers": (1, 2, 3, 4),
        "axu_loss_bet": True,
        **COMMON_FF,
    },
    {
        "name": "05B_count_guided_1234",
        "checkpoint_path": CHECKPOINTS_DIR / "stage_05b_count_guided_betting_from_05a_v1" / "best_eval.pt",
        "bet_multipliers": (1, 2, 3, 4),
        "axu_loss_bet": True,
        **COMMON_FF,
    },
    {
        "name": "06B_decoupled_no_discard_1234",
        "checkpoint_path": CHECKPOINTS_DIR / "stage_06b_decoupled_no_discard_from_05a_v1" / "best_eval.pt",
        "bet_multipliers": (1, 2, 3, 4),
        "axu_loss_bet": True,
        **COMMON_FF,
    },
    {
        "name": "06D_decoupled_ev_cal_1234",
        "checkpoint_path": CHECKPOINTS_DIR / "stage_06d_decoupled_ev_calibration_from_best06b" / "best_eval.pt",
        "bet_multipliers": (1, 2, 3, 4),
        "axu_loss_bet": True,
        **COMMON_FF,
    },
    {
        "name": "07B_spread_123_ev_rank",
        "checkpoint_path": CHECKPOINTS_DIR / "stage_07b_spread_123_ev_rank_from_06d_best" / "best_eval.pt",
        "bet_multipliers": (1, 2, 3),
        "axu_loss_bet": True,
        **COMMON_FF,
    },
    {
        "name": "07C_spread_123_ev_rank_explore",
        "checkpoint_path": CHECKPOINTS_DIR / "stage_07c_spread_123_ev_rank_explore_from_07b" / "best_eval.pt",
        "bet_multipliers": (1, 2, 3),
        "axu_loss_bet": True,
        **COMMON_FF,
    },
]

results = comparison_models(
    checkpoint_specs,
    eval_rounds=5_000,
    eval_max_decisions=60_000,
    device="cpu",
    progress_every_n_rounds=1_000,
    print_summary=True,
)


MODEL RUN
  Slot      : 1/9
  Name      : 03C_playing_only_hard
  Checkpoint: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints\stage_03c_unknown_progress_hard_from_stage03b_v1\best_eval.pt
BLACKJACK CHECKPOINT EVALUATION
  Checkpoint: name=03C_playing_only_hard | path=C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints\stage_03c_unknown_progress_hard_from_stage03b_v1\best_eval.pt
  Model     : arch=feedforward | recurrent=gru | encoder=table_realistic_unknown_progress
  Runtime   : device=cpu | rounds=5000 | max_decisions=60000 | seed=777
  Table     : start=unknown_progress | burned=10-60 | bets=(1,) | pens=(0.75,)
------------------------------------------------------------------------------------------------
PROGRESS 03C_playing_only_hard
  Runtime   : rounds=1000/5000 | decisions=2210
VAL
  Reward  : reward/round=-0.0350 | EV/1000=-34.55 | round

In [6]:
import pandas as pd 

summary = pd.DataFrame([
    {
        "model": r["stage_name"],
        "EV/1000": r["ev_per_1000_hands"],
        "round_std": r["round_std"],
        "win": r["win_frac"],
        "loss": r["loss_frac"],
        "bust": r["bust_frac"],
        "bet_1x": r["bet_1x_frac"],
        "bet_2x": r["bet_2x_frac"],
        "bet_3x": r["bet_3x_frac"],
        "bet_4x": r["bet_4x_frac"],
        "agg_frac": r["agg_frac"],
        "margin": r["bet_q"]["mean_margin_best_aggressive_vs_1x"],}for r in results])

summary.sort_values("EV/1000", ascending=False)

,model,EV/1000,round_std,win,loss,bust,bet_1x,bet_2x,bet_3x,bet_4x,agg_frac,margin
3,05A_count_aux_repr_1234,-29.657643,1.155341,0.431330,0.492436,0.160231,0.9992,0.0000,0.0008,0.0000,0.0008,-0.876357
1,04D_weighted_ce_1234,-37.539620,1.360373,0.430666,0.494255,0.114699,0.9320,0.0242,0.0420,0.0018,0.0438,-0.804603
0,03C_playing_only_hard,-44.178691,1.143570,0.425776,0.498715,0.108915,1.0000,0.0000,0.0000,0.0000,0.0000,0.000000
2,04G_ev_calibrated_1234,-56.528662,1.193136,0.419586,0.504578,0.159236,0.9930,0.0022,0.0048,0.0000,0.0048,-0.982706
7,07B_spread_123_ev_rank,-76.181963,1.223302,0.417561,0.507549,0.096544,0.9994,0.0000,0.0006,0.0000,0.0006,-0.536021
8,07C_spread_123_ev_rank_explore,-79.062376,1.223995,0.415971,0.508542,0.094954,0.9994,0.0000,0.0006,0.0000,0.0006,-0.600493
5,06B_decoupled_no_discard_1234,-86.230876,1.235421,0.414266,0.510630,0.086827,0.9998,0.0000,0.0002,0.0000,0.0002,-0.588357
6,06D_decoupled_ev_cal_1234,-88.036566,1.217888,0.412957,0.512122,0.084459,0.9994,0.0000,0.0006,0.0000,0.0006,-0.520117
4,05B_count_guided_1234,-290.521705,1.449811,0.368379,0.565711,0.332935,0.9994,0.0000,0.0006,0.0000,0.0006,-0.968044


In [7]:
summary.to_csv('Results_models.csv')